# Python script to remove self generated noise by SeaExplorer glider.

This script will: 
1- Calculate the SPL in 1 s intervals for a SeaExplorer recording. 
2- Remove two noise patterns created by the SeaExplorer mechanisms based on the sound of channel 1.
Two samples of the noise need to be stored in a folder called Noise_DIR. 
The files to be analized and cleaned need to be stored in the Mission_DIR folder. 
The software uses spectral matching and the Itakura-Saito distance to detect the desired noise patterns. 
A file segment will be cancelled if it matches one of the two patterns provided and lasts at least one second. 
The length of the file will remain unaltered. The undesired noise segments will be replaced by NaN values. 
A single clean file containing all the files analyzed will be created and saved in the Mission_DIR folder. 
The parameters have been adjusted to remove all "suspicious" segments. 
Script created by Jose A. Diaz, Sept 2026.

In [3]:
import glob
import os
import numpy as np
import scipy.signal as signal
import soundfile as sf


def lpc(signal_in, order):
    """
    Computes Linear Predictive Coding (LPC) coefficients and prediction error variance.
    Matches MATLAB's lpc(x, p) function using the autocorrelation (Levinson-Durbin) method.
    """
    x = np.asarray(signal_in, dtype=np.float64)
    # Autocorrelation via FFT for exact length / correlation
    n = len(x)
    x_padded = np.zeros(2 * n)
    x_padded[:n] = x
    fx = np.fft.fft(x_padded)
    r = np.fft.ifft(fx * np.conj(fx)).real[:order + 1]
    
    # Levinson-Durbin recursion
    a = np.zeros(order + 1, dtype=np.float64)
    a[0] = 1.0
    g = r[0]
    
    if g == 0:
        return a, 0.0
        
    for i in range(1, order + 1):
        gamma = -np.dot(a[:i], r[i:0:-1]) / g
        a[1:i+1] = a[1:i+1] + gamma * a[i-1::-1]
        g = g * (1.0 - gamma ** 2)
        
    return a, g


def smoothdata_movmean(data, window_size):
    """
    Computes moving average smoothing matching MATLAB's smoothdata(x, 'movmean', window_size).
    MATLAB uses endpoint window truncation for symmetric moving average.
    """
    data = np.asarray(data, dtype=np.float64)
    n = len(data)
    out = np.empty(n, dtype=np.float64)
    for idx in range(n):
        # MATLAB symmetrical window around idx:
        # For window_size = W, half_win = (W - 1) / 2
        # For even W, MATLAB places extra point to the left
        left_offset = (window_size - 1) // 2
        right_offset = window_size // 2
        start_idx = max(0, idx - left_offset)
        end_idx = min(n, idx + right_offset + 1)
        out[idx] = np.mean(data[start_idx:end_idx])
    return out


# Noise pattern and file directories
Cur_DIR = os.getcwd()
Noise_DIR = r'/home/jovyan/Data/SeaExplorer/Noise' # There must be two noise patterns in this directory. Edit as needed
Mission_DIR = r'/home/jovyan/Data/SeaExplorer/Sound' # Place the sound files here. Edit as needed

os.chdir(Noise_DIR)
fs_native = 192000 # Assuming that the SeaExplorer sampling rate is 192 KHz
dec = 8 # Decimation factor
fs_processing = 192000 / dec # Downsampled rate (captures up to 12 kHz)
p = 16 # Prediction order

# Load 192kHz noise samples and downsample
# [file,location] = uigetfile('*.wav')
location = Noise_DIR
file = 'Noise1.wav'
FileName = os.path.join(location, file)
x, Fs = sf.read(FileName, dtype='float64', always_2d=False)
xdown = signal.decimate(x, dec, ftype='iir', zero_phase=True)

# Preprocess and obtain noise model A
win = np.hamming(2400)
xdown = xdown[10000:12400]
xdown = xdown - np.mean(xdown)
xdown = xdown * win
xdown = signal.lfilter([1, -0.95], [1.0], xdown)
a1, g1 = lpc(xdown, p)

# Load 192kHz noise samples and downsample
# [file,location] = uigetfile('*.wav')
location = Noise_DIR
file = 'Noise2.wav'
FileName = os.path.join(location, file)
x, Fs = sf.read(FileName, dtype='float64', always_2d=False)
xdown = signal.decimate(x, dec, ftype='iir', zero_phase=True)

# Preprocess and obtain noise model
xdown = xdown[20000:22400] * win
xdown = xdown - np.mean(xdown)
xdown = xdown * win
xdown = signal.lfilter([1, -0.95], [1.0], xdown)
a2, g2 = lpc(xdown, p)

# Pre-calculate target spectral envelopes
nfft = 512
w, hA = signal.freqz(np.sqrt(g1), a1, worN=nfft, fs=fs_processing)
f = w
w, hB = signal.freqz(np.sqrt(g2), a2, worN=nfft, fs=fs_processing)
f = w

TimeWindowMSFD = 0.1 # 100 ms time window

# SeaExplorer parameters
Sh = -153.6
FScal = 4 * np.sqrt(2)
AnalogValueFactorSq = FScal**2 / 4

os.chdir(Mission_DIR)
matched_paths = glob.glob('**/*.wav', recursive=True)
FileList = []
for pth in matched_paths:
    norm_pth = os.path.normpath(pth)
    FileList.append({
        'name': os.path.basename(norm_pth),
        'folder': os.path.abspath(os.path.dirname(norm_pth))
    })
NumFiles = len(FileList)
print(f"NumFiles =\n\n   {NumFiles}\n")
yt = np.array([], dtype=np.float64) # File containing all data samples
YdBt = [] # SPL file
distToAt = [] # Distance to Noise 1 (Sound A)
distToBt = [] # Distance to Noise 2 (Sound B)
WinNum = 1 # Window number;

for k in range(NumFiles):
        Mission_FILE = FileList[k]['name']
        print(f"Mission_FILE =\n\n    '{Mission_FILE}'\n")
        Full_Mission_FILE = os.path.join(FileList[k]['folder'], Mission_FILE)
        FILE_Info = sf.info(Full_Mission_FILE)
        FILE_Info_bin = os.stat(Full_Mission_FILE)
        Fs = FILE_Info.samplerate
        SAMPLES = int(round(TimeWindowMSFD * Fs)) # 1 sample includes all channels

        x, Fs = sf.read(Full_Mission_FILE, dtype='float64', always_2d=True)
        s = x.shape
        y = x[:, 0] # y contains only the 1st channel
        
        START = 1
        i = 1
        YdB = []
        distToA = []
        distToB = []

        while START <= s[0] - SAMPLES + 1:
           yi = y[(START - 1):(START + SAMPLES - 1)]
           YdB.append(10.0 * np.log10(np.mean(yi * yi) * AnalogValueFactorSq) - Sh)

           # Downsample to 24 kHz and preprocess
           yidown = signal.decimate(yi, dec, ftype='iir', zero_phase=True)
           yidown = yidown - np.mean(yidown)
           yidown = yidown * win
           yidown = signal.lfilter([1, -0.95], [1.0], yidown)

           # Compute LPC model 
           aT, gT = lpc(yidown, p)

           # Step C: Generate the live spectral envelope
           w_temp, hT = signal.freqz(np.sqrt(gT), aT, worN=nfft, fs=fs_processing)
           f = w_temp
  
           PA = g1 / (np.abs(hA)**2 + np.finfo(float).eps) # Power spectrum of Sound A
           PB = g2 / (np.abs(hB)**2 + np.finfo(float).eps) # Power spectrum of Sound b
           PT = gT / (np.abs(hT)**2 + np.finfo(float).eps) # power spectrum of current file

           # Itakura-Saito distance to Sound A
           ratio = PT / PA
           is_vector = ratio - np.log(ratio + np.finfo(float).eps) - 1
           distToA.append(np.mean(is_vector))

           # Itakura-Saito distance to Sound B
           ratio = PT / PB
           is_vector = ratio - np.log(ratio + np.finfo(float).eps) - 1
           distToB.append(np.mean(is_vector))

           START = START + SAMPLES
           i = i + 1
           WinNum = WinNum + 1
        yt = np.concatenate([yt, y]) if yt.size > 0 else y.copy() # Total sample file
        YdBt.extend(YdB) # Total SPL vector
        distToAt.extend(distToA) # Distance to A full vector
        distToBt.extend(distToB) # Distance to B full vector

# Smooth distance vectors
distToAt = smoothdata_movmean(distToAt, 6)
distToBt = smoothdata_movmean(distToBt, 6)

contA = 0
contB = 0
NumDet = 0
WinLen = []
WinPos = []

for i in range(1, WinNum):
    idx = i - 1
    if distToAt[idx] < 1.5: # Threshold. Edit if needed
         contA = contA + 1 # Number of detections of A
    else:
       if contA >= 10: # Sound detected during 1 s. Edit if needed
           print('Sound A identified ' + str(contA/10) + ' s')
           NumDet = NumDet + 1
           WinLen.append(contA) # Lenghth of detection window
           WinPos.append(i - contA) # Position of window
       contA = 0

    if distToBt[idx] < 1.5: # Threshold. Edit if needed
       contB = contB + 1
    else:
       if contB >= 10: # Sound detected during 1 s. Edit if needed
           print('Sound B identified ' + str(contB/10) + ' s')
           NumDet = NumDet + 1
           WinLen.append(contB) # Lenghth of detection window
           WinPos.append(i - contB) # Position of window
       contB = 0

# Remove undesired noise segments
for i in range(1, NumDet + 1):
    det_idx = i - 1
    iini = int(round((WinPos[det_idx] - 1) * fs_native / 10 + 1))
    ifin = int(round((WinPos[det_idx] - 1 + WinLen[det_idx]) * fs_native / 10))
    yt[(iini - 1):ifin] = np.nan

# Save full clean file
sf.write('FullFileClean.wav', yt, int(Fs))
os.chdir(Cur_DIR)


NumFiles =

   2

Mission_FILE =

    'FullFile2.wav'

Mission_FILE =

    'FullFile1.wav'

Sound B identified 1.2 s
Sound B identified 6.7 s
Sound B identified 1.8 s
Sound B identified 2.9 s
Sound B identified 1.9 s
Sound B identified 1.7 s
Sound B identified 3.2 s
Sound B identified 6.2 s
Sound B identified 2.6 s
Sound A identified 41.5 s
Sound B identified 2.9 s
Sound B identified 2.0 s
Sound B identified 1.1 s
Sound B identified 4.0 s
Sound B identified 6.5 s
Sound B identified 1.1 s
Sound B identified 12.9 s
Sound B identified 1.1 s
Sound A identified 25.8 s
Sound B identified 6.9 s
Sound B identified 1.0 s
Sound B identified 2.2 s
Sound B identified 2.7 s
Sound A identified 41.8 s
Sound B identified 6.7 s
Sound B identified 3.0 s
Sound A identified 25.3 s
Sound B identified 20.9 s
Sound B identified 2.1 s
